<a href="https://colab.research.google.com/github/julienessan/Gomycode_Cours_Data_Sciences/blob/main/Checkpoint_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Checkpoint: Web Scraping**

**Objectif**

L'objectif  est d'automatiser l'extraction du contenu HTML, des titres d'articles, du texte et des liens internes des pages Wikipédia dans une fonction consolidée qui accepte n'importe quelle URL Wikipédia pour un traitement efficace des données.

In [4]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [15]:
#Ecrire une fonction pour récupérer et analyser le contenu HTML d'une page Wikipédia
def get_content(url):
    headers = { "User-Agent" : "CheckpointWebScraping/1.0 (julienessan@yahoo.fr)" }
    reponse = requests.get(url, headers=headers)
    soup = BeautifulSoup(reponse.content, 'html.parser')
    soup.prettify()
    return soup


In [16]:
get_content("https://fr.wikipedia.org/wiki/NoSQL")

<!DOCTYPE html>

<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-feature-navigation-update-disabled vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="fr">
<head>
<meta charset="utf-8"/>
<title>NoSQL — Wikipédia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feat

In [17]:
#Ecrire une fonction pour extraire le titre de l'article
def get_title(soup):
    titre_article = soup.find(id="firstHeading").get_text()
    return titre_article

In [18]:
get_title(soup)

'NoSQL'

In [19]:
# Ecrire une fonction pour extraire le texte de l'article pour chaque paragraphe avec leurs
# en-têtes respectifs. Mettez en correspondance ces titres avec leurs paragraphes respectifs dans le dictionnaire.
def get_text(soup):
    titre_paragraphe = []
    texte_article = []

    main_content_div = soup.find('div', class_='mw-parser-output')
    if not main_content_div:
        return [], []

    elements = main_content_div.find_all(['h3', 'p', 'ul'])

    # Initialisation pour la section d'introduction (avant le premier h3)
    current_section_title = "Introduction"
    current_section_paragraphs = []

    for element in elements:
        if element.name == 'h3':
            # Enregistrer la section actuelle avant d'en commencer une nouvelle
            # À ajouter uniquement s'il y avait effectivement du contenu ou s'il ne s'agit pas de la section initiale « Introduction » et qu'elle contient du contenu
            if current_section_paragraphs or (current_section_title != "Introduction" and current_section_title is not None):
                titre_paragraphe.append(current_section_title)
                texte_article.append('\n'.join(current_section_paragraphs))
            elif current_section_title == "Introduction" and not current_section_paragraphs:
                # Si « Introduction » a été défini mais qu'aucun contenu n'a été trouvé, ne pas encore ajouter de section d'introduction vide
                pass

            # Commencer une nouvelle section avec ce titre h3
            current_section_title = element.get_text(strip=True)
            current_section_paragraphs = [] # Reset paragraphs for the new section
        elif element.name == 'p':
            # Ajouter un paragraphe à la section actuelle
            paragraph_text = element.get_text(strip=True)
            if paragraph_text:
                current_section_paragraphs.append(paragraph_text)
        elif element.name == 'ul':
            # Ajouter des éléments à la liste de la section actuelle
            list_items = element.find_all('li')
            for item in list_items:
                item_text = item.get_text(strip=True)
                if item_text:
                    current_section_paragraphs.append(f'• {item_text}')

    # À la fin de la boucle, ajouter la dernière section collectée
    if current_section_paragraphs or (current_section_title is not None and current_section_title != "Introduction"):
        titre_paragraphe.append(current_section_title)
        texte_article.append('\n'.join(current_section_paragraphs))
    # Traitement particulier d'une « Introduction » susceptible de contenir du contenu
    elif current_section_title == "Introduction" and current_section_paragraphs:
        titre_paragraphe.append(current_section_title)
        texte_article.append('\n'.join(current_section_paragraphs))

    return pd.DataFrame({'Titre': titre_paragraphe, 'Texte': texte_article})

In [20]:
get_text(soup)

,Titre,Texte
0,Introduction,"Eninformatiqueet enbases de données,NoSQLdésig..."
1,Domination historique des SGBD relationnels,Les SGBD relationnels créés dans les années 19...
2,Pionniers du modèle NoSQL,Ce sont les grandes entreprises du web amenées...
3,Invention et popularisation du terme NoSQL,La parution d'articles présentant ces systèmes...
4,Convention NoSQL de 2009,Plus de cent développeurs de logiciels ont ass...
5,NoSQL orienté-agrégats,Une des caractéristiques retrouvées dans de no...
6,NoSQL orienté-graphes,Lesbases de données orientées graphepermettent...
7,NoSQL sans-schéma,La première étape de la création d'une base de...
8,Autres,Lescapacités ACIDgarantissent que si plusieurs...
9,Articles connexes,• Base de données orientée objet\n• Base de do...


In [13]:
import re
#Ecrivez une fonction pour collecter tous les liens qui redirigent vers une autre page Wikipedia.
def get_wikipedia_links(soup):
  links = []
  main_content_div = soup.find('div', class_='mw-parser-output')
  if not main_content_div:
      return links

  for link in main_content_div.find_all('a', href=True):
        href = link['href']
        # Vérifiez s'il s'agit d'un lien interne à Wikipédia en recherchant « /wiki/ »
        if '/wiki/' in href:
            # Extraire la partie de l'URL *après* « /wiki/ »
            wiki_path_segment = href.split('/wiki/', 1)[1]

            # Vérifiez maintenant si ce segment contient deux points.
            # Les deux points indiquent généralement ici des pages spéciales (Fichier:, Catégorie:, etc.)
            # que nous souhaitons exclure.
            if ':' not in wiki_path_segment:
                links.append(href)
  return links

In [14]:
get_wikipedia_links(soup)

['https://fr.wikipedia.org/wiki/Informatique',
 'https://fr.wikipedia.org/wiki/Base_de_données',
 'https://fr.wikipedia.org/wiki/Système_de_gestion_de_base_de_données',
 'https://fr.wikipedia.org/wiki/Paradigme',
 'https://fr.wikipedia.org/wiki/Base_de_données_relationnelle',
 'https://fr.wikipedia.org/wiki/Pramod_J._Sadalage?action=edit&redlink=1',
 'https://fr.wikipedia.org/wiki/Martin_Fowler',
 'https://fr.wikipedia.org/wiki/Centre_de_données',
 'https://fr.wikipedia.org/wiki/Infrastructure',
 'https://fr.wikipedia.org/wiki/Grappe_de_serveurs',
 'https://fr.wikipedia.org/wiki/Programmation_concurrente',
 'https://fr.wikipedia.org/wiki/Propriétés_ACID',
 'https://fr.wikipedia.org/wiki/Base_de_données_relationnelle',
 'https://fr.wikipedia.org/wiki/Base_de_données_orientée_objet',
 'https://fr.wikipedia.org/wiki/Base_de_données_hiérarchique',
 'https://fr.wikipedia.org/wiki/Système_de_gestion_de_base_de_données_relationnel-objet',
 'https://fr.wikipedia.org/wiki/Grappe_de_serveurs',
 

In [21]:
#Regroupez toutes les fonctions précédentes en une seule fonction qui prend en paramètre un lien Wikipedia.
def get_wikipedia_content(url):
    soup = get_content(url)
    titres = get_title(soup)
    textes = get_text(soup)
    liens = get_wikipedia_links(soup)
    return titres, textes, liens

In [23]:
get_wikipedia_content("https://fr.wikipedia.org/wiki/Logiciel")

('Logiciel',
                Titre                                              Texte
 0       Introduction  Ne doit pas être confondu avecProgramme inform...
 1        Années 1950  Dans les années 1950 les logiciels sont écrits...
 2        Années 1960  Les sociétés de logiciel se multiplient dans l...
 3        Années 1980  Dans les années 1980, la production / consomma...
 4        Années 1990  En 1990, il existe auxÉtats-Unis, premier cons...
 5   Philosophielibre  Lelogiciel libreest un mouvement social basée ...
 6   Union européenne  Dans l'Union européenne, unalgorithmene peut ê...
 7             Bogues  Lesbogues, oubugs, sont des erreurs de concept...
 8  Articles connexes  • Logiciel en tant que service\n• Catégorie:Lo...
 9     Liens externes  • Ressource relative à la santé:Medical Subjec...,
 ['https://fr.wikipedia.org/wiki/Programme_informatique',
  'https://fr.wikipedia.org/wiki/Application_(informatique)',
  "https://fr.wikipedia.org/wiki/Capture_d'écran",
  'https://f